## Secure Multiparty Computation

### Workflow of the demo:
1. Input Balances: The program begins by defining account balances for multiple institutions.

2. Share Generation: Using Shamir’s Secret Sharing, the balances are split into n random shares.

3. Collaboration: A random subset of institutions (equal to the threshold) is chosen to combine their shares for reconstruction.

4. Reconstruction: Lagrange interpolation is applied to recover the total balance securely.

5. Privacy and Security: Gaussian noise ensures additional protection, making it harder for adversaries to reverse-engineer the original balances from fewer shares.

Here the code generates polynomial-based shares for each financial institution's account balance. It creates a random polynomial of degree threshold - 1 where the constant term is the account balance. Each share is a point on this polynomial, and Gaussian noise is added for differential privacy.

In [15]:
import random

# Function to generate polynomial shares using Shamir's Secret Sharing
def generate_shares(account_balances, n, threshold):
    shares = []
    for institution_id, balance in account_balances.items():
        # Generating a random polynomial of degree (threshold - 1) with the constant term as the balance
        coefficients = [random.randint(1, 1000) for _ in range(threshold - 1)] + [balance]

        # Generating n shares for the polynomial
        institution_shares = []
        for i in range(1, n + 1):
            # Evaluating the polynomial at x=i
            y_value = sum(coeff * (i ** exp) for exp, coeff in enumerate(reversed(coefficients)))
            # Adding Gaussian noise for privacy
            noisy_y_value = int(y_value + random.gauss(0, 10))
            institution_shares.append((i, noisy_y_value))
        
        shares.append((institution_id, institution_shares))
    return shares


The following function reconstructs the original total balance using a subset of shares. Lagrange interpolation calculates the polynomial's value at x=0 (the secret), ensuring reconstruction only if the threshold number of shares is available.

In [16]:

# Function to reconstruct the total balance from shares using Lagrange interpolation
def reconstruct_balance(used_shares):
    total_balance = 0
    for institution_id, shares in used_shares:
        # Using only the first t shares for reconstruction
        t = len(shares)
        x_vals, y_vals = zip(*shares)
        reconstructed = 0

        for j in range(t):
            # Lagrange basis polynomial calculation
            numerator, denominator = 1, 1
            for m in range(t):
                if m != j:
                    numerator *= -x_vals[m]
                    denominator *= (x_vals[j] - x_vals[m])
            reconstructed += y_vals[j] * (numerator / denominator)
        
        total_balance += int(reconstructed)
    return total_balance



The following main function demonstrates the entire workflow:

 - Generate Shares: It splits each balance into n shares with Shamir's Secret Sharing.
 
 - Collaboration: Randomly selects shares from participating institutions.
 
 - Reconstruction: Combines the selected shares to calculate the total account balance.

In [17]:
def main():
    # Financial institution account balances
    account_balances = {
        "Bank A": 10000,
        "Bank B": 1500,
        "Bank C": 1000,
        "Bank D": 5000
    }

    # Parameters
    n = 5  # Number of total shares
    threshold = 3  # Threshold for reconstruction

    # Generating shares using Shamir's Secret Sharing
    shares = generate_shares(account_balances, n, threshold)
    print("Generated Shares (per institution):")
    for institution_id, institution_shares in shares:
        print(f"{institution_id}: {institution_shares}")

    # Simulating collaboration between institutions
    used_shares = random.sample(shares, threshold)
    collaboration_shares = [(institution_id, random.sample(institution_shares, threshold)) for institution_id, institution_shares in used_shares]

    print("\nShares used for collaboration:")
    for institution_id, institution_shares in collaboration_shares:
        print(f"{institution_id}: {institution_shares}")

    # Reconstruction of the total balance
    reconstructed_total_balance = reconstruct_balance(collaboration_shares)
    print("\nReconstructed Total Account Balance:", reconstructed_total_balance)

    # Checking if the total account balance exceeds a certain threshold
    balance_threshold = 10000
    if reconstructed_total_balance > balance_threshold:
        print(f"Total account balance exceeds the threshold of {balance_threshold}.")
    else:
        print(f"Total account balance does not exceed the threshold of {balance_threshold}.")

if __name__ == "__main__":
    main()


Generated Shares (per institution):
Bank A: [(1, 10534), (2, 11383), (3, 12441), (4, 13802), (5, 15398)]
Bank B: [(1, 2476), (2, 5150), (3, 9548), (4, 15627), (5, 23437)]
Bank C: [(1, 1787), (2, 3235), (3, 5333), (4, 8066), (5, 11465)]
Bank D: [(1, 5261), (2, 5994), (3, 7204), (4, 8848), (5, 10955)]

Shares used for collaboration:
Bank B: [(2, 5150), (1, 2476), (3, 9548)]
Bank A: [(4, 13802), (3, 12441), (2, 11383)]
Bank D: [(2, 5994), (1, 5261), (3, 7204)]

Reconstructed Total Account Balance: 16707
Total account balance exceeds the threshold of 10000.
